# Курсовая — Полный пайплайн анализа видеоконтента

Автоматизированный анализ видеофайла для выявления деструктивного контента.

Шаги:
1. Загрузка видео из Google Drive
2. Нарезка на кадры
3. OCR субтитров + дедупликация
4. YOLO детекция + склейка событий
5. Whisper транскрипция аудио
6. Итоговый JSON отчёт

In [ ]:
!pip install opencv-python-headless easyocr ultralytics openai-whisper moviepy rapidfuzz torch torchvision tqdm -q
!apt-get install -y ffmpeg -q

In [ ]:
from google.colab import drive, files
import os, json, shutil, torch

drive.mount('/content/drive', force_remount=True)

BASE_DRIVE  = '/content/drive/MyDrive/cv-frames'
VIDEO_DIR   = f'{BASE_DRIVE}/видио'
FRAMES_DIR  = f'{BASE_DRIVE}/кадры'
RESULTS_DIR = f'{BASE_DRIVE}/результаты'

for d in [VIDEO_DIR, FRAMES_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'устройство: {DEVICE}')

In [ ]:
FRAME_STEP       = 30
CONF_YOLO        = 0.4
OCR_DEDUP_THR    = 85
DET_MERGE_WINDOW = 5

In [ ]:
existing = [f for f in os.listdir(VIDEO_DIR) if f.endswith(('.mp4', '.avi', '.mkv'))]

if existing:
    print(f'видео в drive: {existing}')
    VIDEO_PATH = f'{VIDEO_DIR}/{existing[0]}'
else:
    uploaded = files.upload()
    name = list(uploaded.keys())[0]
    VIDEO_PATH = f'{VIDEO_DIR}/{name}'
    shutil.move(name, VIDEO_PATH)

print(f'используем: {VIDEO_PATH}')

In [ ]:
import cv2
from tqdm.notebook import tqdm

cap   = cv2.VideoCapture(VIDEO_PATH)
fps   = cap.get(cv2.CAP_PROP_FPS)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

frame_idx = saved = 0
for _ in tqdm(range(total), desc='нарезка'):
    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx % FRAME_STEP == 0:
        cv2.imwrite(f'{FRAMES_DIR}/frame_{frame_idx:06d}.jpg', frame)
        saved += 1
    frame_idx += 1
cap.release()

FRAME_LIST = sorted(f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg'))
print(f'кадров: {saved}, fps: {fps:.1f}, длина: {total/fps:.1f} сек')

In [ ]:
import easyocr
import pandas as pd
from rapidfuzz import fuzz, process

reader   = easyocr.Reader(['ru', 'en'], gpu=(DEVICE == 'cuda'))
ocr_rows = []

for fname in tqdm(FRAME_LIST, desc='ocr'):
    img = cv2.imread(f'{FRAMES_DIR}/{fname}')
    if img is None:
        continue
    h    = img.shape[0]
    crop = img[int(h * 0.85):, :]
    for _, text, prob in reader.readtext(cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)):
        if text.strip() and prob > 0.4:
            ocr_rows.append({'frame': fname, 'text': text.strip(), 'conf': round(prob, 3)})

df_ocr = pd.DataFrame(ocr_rows)
df_ocr.to_csv(f'{RESULTS_DIR}/ocr_results.csv', index=False, encoding='utf-8-sig')

def dedup(texts, thr=OCR_DEDUP_THR):
    uniq = []
    for t in texts:
        best = process.extractOne(t, uniq, scorer=fuzz.ratio) if uniq else None
        if best is None or best[1] < thr:
            uniq.append(t)
    return uniq

uniq_subs = dedup(df_ocr['text'].tolist())
print(f'ocr: {len(df_ocr)} строк -> {len(uniq_subs)} уникальных')

In [ ]:
from ultralytics import YOLO

yolo      = YOLO('yolov8n.pt')
yolo_rows = []

for fname in tqdm(FRAME_LIST, desc='yolo'):
    for r in yolo(f'{FRAMES_DIR}/{fname}', conf=CONF_YOLO, verbose=False):
        for box in r.boxes:
            yolo_rows.append({
                'frame':     fname,
                'frame_num': int(fname.split('_')[1].split('.')[0]),
                'class':     yolo.names[int(box.cls[0])],
                'conf':      round(float(box.conf[0]), 3),
            })

df_yolo = pd.DataFrame(yolo_rows)
df_yolo.to_csv(f'{RESULTS_DIR}/yolo_detections.csv', index=False)

def merge_det(g):
    g = g.sort_values('frame_num')
    evts, start, prev = [], g.iloc[0], g.iloc[0]['frame_num']
    for _, r in g.iloc[1:].iterrows():
        if r['frame_num'] - prev > DET_MERGE_WINDOW:
            evts.append({'class': start['class'], 'start_frame': start['frame_num'], 'end_frame': prev})
            start = r
        prev = r['frame_num']
    evts.append({'class': start['class'], 'start_frame': start['frame_num'], 'end_frame': prev})
    return pd.DataFrame(evts)

df_merged = df_yolo.groupby('class', group_keys=False).apply(merge_det).reset_index(drop=True)
df_merged.to_csv(f'{RESULTS_DIR}/yolo_merged.csv', index=False)
print(f'yolo: {len(df_yolo)} детекций -> {len(df_merged)} событий')

In [ ]:
import whisper
from moviepy.editor import VideoFileClip

audio_path = '/content/audio.wav'
VideoFileClip(VIDEO_PATH).audio.write_audiofile(audio_path, verbose=False, logger=None)

w_model    = whisper.load_model('base')
transcript = w_model.transcribe(audio_path, language='ru', verbose=False)

segments = [
    {'start': round(s['start'], 2), 'end': round(s['end'], 2), 'text': s['text'].strip()}
    for s in transcript['segments']
]
print(f'whisper: {len(segments)} сегментов')

In [ ]:
report = {
    'video':           os.path.basename(VIDEO_PATH),
    'duration_sec':    round(total / fps, 1),
    'frames_analyzed': len(FRAME_LIST),
    'ocr': {
        'total':  len(df_ocr),
        'unique': len(uniq_subs),
        'sample': uniq_subs[:10],
    },
    'yolo': {
        'total_detections': len(df_yolo),
        'merged_events':    len(df_merged),
        'top_classes':      df_yolo['class'].value_counts().head(5).to_dict(),
        'events':           df_merged.to_dict(orient='records'),
    },
    'whisper': {
        'total_segments': len(segments),
        'preview':        transcript['text'][:500],
        'segments':       segments[:10],
    },
}

report_path = f'{RESULTS_DIR}/final_report.json'
with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print(f'отчёт: {report_path}')

In [ ]:
files.download(report_path)